# **VALIDATION RULES**

## Import data

In [1]:
#Import data from database
from pathlib import Path
from sqlalchemy import create_engine
import pandas as pd

# Current working directory (notebook folder)
base_dir = Path.cwd()

# Build path to database
db_path = (
    base_dir
    / "sewer_database_example.db"
).resolve()


engine = create_engine(f"sqlite:///{db_path}")

df_pipes = pd.read_sql_table("pipe", engine)
print(f"Loaded {len(df_pipes)} rows from 'PIPES'.")

df_hydraulic = pd.read_sql_table("hydraulic_properties", engine)
print(f"Loaded {len(df_hydraulic)} rows from 'HYDRAULIC_PROPERTIES'.")

df_cctv = pd.read_sql_table("inspection", engine)
print(f"Loaded {len(df_cctv)} rows from 'CCTV'.")

df_defects = pd.read_sql_table("defect", engine)
print(f"Loaded {len(df_defects)} rows from 'DEFECTS'.")

Loaded 2984 rows from 'PIPES'.
Loaded 2984 rows from 'HYDRAULIC_PROPERTIES'.
Loaded 2776 rows from 'CCTV'.
Loaded 42278 rows from 'DEFECTS'.


## Preprocessing

In [75]:
from data_preprocessing import drop_all_nan_columns, filter_valid_defects, remove_uncompleted_without_defects

In [76]:
# Removes columns from database that are not being used
df_pipes, df_hydraulic, df_cctv, df_defects = drop_all_nan_columns(df_pipes, df_hydraulic, df_cctv, df_defects)

Columns with NaN values in all rows were deleted from dataframes.


In [77]:
# Keeps only defects and removes other observations (features)

# Dictionary (1 = valid defect, 0 = non-defect / feature)
def_or_fea_map = {
    "IS": 0, "WL": 0, "LO": 0, "DG": 1, "GP": 0, "LB": 0, "OT": 1,
    "IA": 0, "MHJ": 1, "JF": 1, "ED": 1, "GC": 0, "IE": 0, "L": 0,
    "JD": 1, "RI": 1, "SD": 1, "LF": 1, "MC": 0, "CF": 0, "CC": 1,
    "LP": 1, "LD": 0, "IP": 1, "DP": 1, "PH": 1, "JO": 1, "CL": 1,
    "OP": 1, "DE": 1, "LX": 1, "CM": 1, "SV": 1, "PF": 1, "PB": 1,
    "TM": 1, "DF": 1, "DC": 0, "LOV": 0, "PR": 0, "PL": 1, "B": 1, "EX": 1
}

df_defects = filter_valid_defects(df_defects,def_or_fea_map)

The features were remove. Now, df_defects has 31137 defects.


In [78]:
df_cctv = remove_uncompleted_without_defects(df_cctv, df_defects)

9 uncompleted inspections without defects were removed


## Validation rules of pipes

In [79]:
from count_changes import evaluate_rule_change

from validation_rules import remove_nan_pipe_id, remove_duplicate_pipe_id, clean_material_column,filter_installation_year, reclassify_sewer_category, replace_zero_diameter_with_nan, set_diameter_outliers_to_nan, set_small_diameters_to_nan, set_small_pipe_length_to_nan, set_negative_slope_to_nan,  set_invalid_invert_difference_to_nan, set_high_slope_to_nan, set_negative_depths_to_nan, set_excessive_depth_to_nan, set_non_positive_flows_to_nan

### Pipes ID

In [80]:
# Removes pipes without ID
df_pipes = remove_nan_pipe_id(df_pipes)

0 pipes were removed due to missing Pipe_ID.


In [81]:
# Removes duplicate pipes
df_pipes = remove_duplicate_pipe_id(df_pipes)

0 duplicate pipes were removed based on Pipe_ID.


### Material

In [82]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

# Converts to NaN
df_pipes = clean_material_column(df_pipes)

In [83]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Material',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 295
Changed with CCTV: 277


### Installation year

In [84]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

# Dictionary with Installation year valid ranges according to the pipe material
material_year_limits = {
    "VC":  {"min": 1880},
    "AC":  {"min": 1920, "max": 2016},
    "GRP": {"min": 1970},
    "PE":  {"min": 1985},
    "CONC":{"min": 1910},
    "PVC": {"min": 1950},
    "CI":  {"min": 1880},
    "DI":  {"min": 1955},
    "FRP": {"min": 1970}
}

# Convert to NaN installation years out of valid ranges
df_pipes["Installation_year"] = df_pipes.apply(
    filter_installation_year,
    axis=1,
    limits_dict=material_year_limits
)

In [85]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Installation_year',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 308
Changed with CCTV: 287


### Sewer category

In [86]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#Transform transmission pipes to local when the diameter is less than or equal to a threshold diameter
df_pipes = reclassify_sewer_category(df_pipes, threshold_diameter=100)

In [87]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Sewer_category',
    df_cctv=df_cctv,
    change_type='value_change'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 0
Changed with CCTV: 0


### Diameter

In [88]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#Transform diameters equal to 0 to NaN
df_pipes = replace_zero_diameter_with_nan(df_pipes)

In [89]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Diameter',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 0
Changed with CCTV: 0


In [90]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

# Convert to NaN diameters exceeding a reasonable upper limit according to the material

# Dictionary with max. diameters
diameter_outlier_max = {
    "AC": 600,
    "CONC": 3600,
    "VC": 675,
    "PVC": 1500,
    "PE": 3200
}

# Apply validation rule
df_pipes = set_diameter_outliers_to_nan(df_pipes, diameter_outlier_max)

In [91]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Diameter',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 34
Changed with CCTV: 30


In [92]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#Transform diameters smaller than a minimum diameter to NaN
df_pipes = set_small_diameters_to_nan(df_pipes, min_diameter=100)

In [93]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Diameter',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 0
Changed with CCTV: 0


### Length

In [94]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#Transform lengths that are less than a minimum length to NaN
df_pipes = set_small_pipe_length_to_nan(df_pipes, min_length=0.6)

In [95]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Pipe_length',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 0
Changed with CCTV: 0


### Slope

In [96]:
# Calculate slope

# Calculate the difference between invert levels
df_pipes["Dif_invert"] = df_pipes["UP_depth"] - df_pipes["DW_depth"]

# Calculate the slope with the
df_pipes['Slope'] = (df_pipes["Dif_invert"]/df_pipes["Pipe_length"])*100

In [97]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

# When the slope is negative or equal to zero, transform slopes, invert levels, and depths to NaN
df_pipes = set_negative_slope_to_nan(df_pipes)

In [98]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Slope',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 2404
Changed with CCTV: 2210


In [99]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

# When the difference between the inverts is greater than the length of the pipe, transform slopes, invert levels, and depths to NaN
df_pipes = set_invalid_invert_difference_to_nan(df_pipes)

In [100]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Slope',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 0
Changed with CCTV: 0


In [101]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#When the slope is greater than a maximum value, transform slopes, invert levels, and depths to NaN
df_pipes = set_high_slope_to_nan(df_pipes, max_slope=30)

In [102]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Slope',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 1
Changed with CCTV: 1


In [103]:
# Drop temporary column Dif_invert
df_pipes = df_pipes.drop("Dif_invert", axis=1)

### Depth

In [104]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#Transform invert levels and depths to NaN when the depth is a negative value
df_pipes = set_negative_depths_to_nan(df_pipes)

In [105]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Depth',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 0
Changed with CCTV: 0


In [106]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#When the average depth is greater than a maximum value, transform slopes, invert levels, and depths to NaN
df_pipes = set_excessive_depth_to_nan(df_pipes, max_depth=15)

In [107]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Depth',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 1
Changed with CCTV: 1


## Validation rules of Hydraulic Properties

### Flow rate

In [108]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_hydraulic.copy()

# Convert to NaN the flow rates that are <= 0 (dry and wet weather)
df_hydraulic = set_non_positive_flows_to_nan(df_hydraulic)

In [109]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_hydraulic,
    column='Dry_peak_flow_rate',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 0
Changed with CCTV: 0


In [110]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_hydraulic,
    column='Wet_peak_flow_rate',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

Total changed: 0
Changed with CCTV: 0


 # **DOWNLOAD DATA**

In [111]:
output_path = "Validated_data.xlsx"

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    df_pipes.to_excel(writer, sheet_name="PIPES", index=False)
    df_hydraulic.to_excel(writer, sheet_name="HYDRAULIC_PROPERTIES", index=False)
    df_cctv.to_excel(writer, sheet_name="CCTV", index=False)
    df_defects.to_excel(writer, sheet_name="DEFECTS", index=False)
print("✅ Finished successfully")

✅ Finished successfully
